In [2]:
import pandas as pd
import time
from dataLoader import load_msi_data, HSI_SATELLITE_DIR, gt_train_df, column_names

from MultiOutputTuner import MultiOutputTuner, score
from joblib import parallel_backend

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.multioutput import MultiOutputRegressor

from interpret import show
from interpret.blackbox import ShapKernel, LimeTabular

X = load_msi_data(HSI_SATELLITE_DIR)
y = gt_train_df[column_names].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8)

In [8]:
from xgboost import XGBRegressor

param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.05, 0.1, 0.15],
    "model__subsample": [0.5, 0.7, 0.9],
    "model__colsample_bytree": [0.5, 0.8, 1.0],
    # "model__min_child_weight": [3, 5, 7],
    "model__gamma": [0.3, 0.5, 0.7],
}

xgb = MultiOutputTuner(
    XGBRegressor(random_state=42), 
    param_grid, 
    cv=5, 
    verbose=1, 
    n_jobs=-1, 
    scoring="neg_mean_squared_error",
    reduce = 0.95)
start = time.perf_counter()
with parallel_backend("threading"):
    xgb.fit(X, y)
end = time.perf_counter()

print(f"time: {end-start:4f}")
xgb.save("models/hsi/xgb_95")

Fitting 5 folds for each of 729 candidates, totalling 3645 fits
Fitting 5 folds for each of 729 candidates, totalling 3645 fits
Fitting 5 folds for each of 729 candidates, totalling 3645 fits
Fitting 5 folds for each of 729 candidates, totalling 3645 fits
Fitting 5 folds for each of 729 candidates, totalling 3645 fits
Fitting 5 folds for each of 729 candidates, totalling 3645 fits
Column: 2 	 score: -0.044573952674316554 	 params: {'model__colsample_bytree': 1.0, 'model__gamma': 0.3, 'model__learning_rate': 0.1, 'model__max_depth': 7, 'model__n_estimators': 100, 'model__subsample': 0.5}
Column: 3 	 score: -0.18424300431751978 	 params: {'model__colsample_bytree': 1.0, 'model__gamma': 0.7, 'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 50, 'model__subsample': 0.5}
Column: 1 	 score: -3.9013651895870174 	 params: {'model__colsample_bytree': 1.0, 'model__gamma': 0.3, 'model__learning_rate': 0.15, 'model__max_depth': 3, 'model__n_estimators': 50, 'model__subsamp

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

param_grid = {
    "model__n_estimators": [100, 150, 200],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [2, 5, 7],
    "model__min_samples_split": [2, 10, 20],
    "model__loss": ["squared_error", "absolute_error"]
}

model = MultiOutputTuner(
    GradientBoostingRegressor(random_state = 42), 
    param_grid, 
    scoring="neg_mean_squared_error",
    cv=5, 
    verbose=1,
    n_jobs=-1,
    reduce = 0.9
)

start = time.perf_counter()
with parallel_backend("threading"):
    model.fit(X, y)
end = time.perf_counter()

print(f"time: {end-start:4f}")
model.save("models/hsi/gradientBoosting_9")

Fitting 5 folds for each of 162 candidates, totalling 810 fits
Fitting 5 folds for each of 162 candidates, totalling 810 fits
Fitting 5 folds for each of 162 candidates, totalling 810 fits
Fitting 5 folds for each of 162 candidates, totalling 810 fits
Fitting 5 folds for each of 162 candidates, totalling 810 fits
Fitting 5 folds for each of 162 candidates, totalling 810 fits


In [5]:
score(X_train, y_train, X_test, y_test, model)

Baseline mse: [3.66803272e+03 4.44196617e+00 7.14399101e-02 3.28052784e-01
 2.02061895e+02 5.69132916e+02]
Class Fe baseline score: 1.2213678620528075
Class Zn baseline score: 1.1114216725128663
Class B baseline score: 1.2370883225361424
Class Cu baseline score: 1.4935941949658817
Class S baseline score: 1.0519303776395308
Class Mn baseline score: 1.226879813566385
Final baseline score: 1.223713707212269
Model mse: [1.31539625e+03 1.20188595e+00 2.00339399e-02 9.95334568e-02
 9.55483732e+01 2.14730468e+02]
Final testing model score: [0.43799574 0.30072316 0.34691747 0.45316669 0.49742301 0.46289447]


In [ ]:
from sklearn.neighbors import KNeighborsRegressor

param_grid = {
    "model__n_neighbors": [5, 7, 9, 11, 13, 15],
    "model__weights": ["uniform", "distance"],
    "model__algorithm": ["auto", "ball_tree", "kd_tree", "brute"],
    "model__leaf_size": [2, 5, 8, 10, 12, 15, 18, 20],
    "model__p": [1, 2, 3, 4, 5, 6, 7],
    "model__metric": ["minkowski", "euclidean", "manhattan", "chebyshev"]
}
kNeighbors = MultiOutputTuner(
    KNeighborsRegressor(), 
    param_grid, 
    cv=5,
    n_jobs=-1,
    verbose = 1,
    reduce=0.9
)

start = time.perf_counter()
with parallel_backend("threading"):
    kNeighbors.fit(X_train, y_train)
end = time.perf_counter()

print(f"time: {end-start:4f}")
kNeighbors.save("models/hsi/kNeighbors_9")

In [ ]:
from sklearn.ensemble import AdaBoostRegressor

param_grid = {
    "model__n_estimators": [100, 200, 300, 400, 500],
    "model__learning_rate": [0.01, 0.02, 0.03, 0.04, 0.05 ],
    "model__loss": ["linear", "square", "exponential"]
}

adaBoost = MultiOutputTuner(
    AdaBoostRegressor(random_state = 42), 
    param_grid, 
    cv=5, 
    verbose=1,
    n_jobs=-1,
    reduce = 0.975
)

start = time.perf_counter()
with parallel_backend("threading"):
    adaBoost.fit(X_train, y_train)
end = time.perf_counter()

print(f"time: {end-start:4f}")
adaBoost.save("models/hsi/adaBoost_9")

In [3]:
# Save results

In [8]:
from sklearn.decomposition import PCA
import pandas as pd
from dataLoader import *

X_train = load_msi_data(HSI_SATELLITE_DIR) 
# pca = PCA(n_components=0.95)
pca.fit(X_train)

X_test = load_msi_data(HSI_SATELLITE_TEST_DIR)
# X_test = pca.transform(X_test)

xgb = MultiOutputTuner.load('models/hsi/xgb_95')
gradientBoosting = MultiOutputTuner.load('models/hsi/gradientBoosting_95')
kneighbours = MultiOutputTuner.load('models/hsi/kNeighbors_95')
# ebm = MultiOutputTuner.load('models/hsi/ebm')
adaBoost = MultiOutputTuner.load('models/hsi/adaBoost_95')
randomforest = MultiOutputTuner.load('models/hsi/randomForest')

xgb_predictions = xgb.predict(X_test)
pd.DataFrame(data=xgb_predictions, columns=column_names).to_csv("xgb_95.csv", index_label="sample_index")

gradient_predictions = gradientBoosting.predict(X_test)
pd.DataFrame(data=gradient_predictions, columns=column_names).to_csv("gradient_95.csv", index_label="sample_index")

kneighbours_predicrions= kneighbours.predict(X_test)
pd.DataFrame(data=kneighbours_predicrions, columns=column_names).to_csv("kneighbors_95.csv", index_label="sample_index")

randomforest_predictions = randomforest.predict(X_test)
pd.DataFrame(data=randomforest_predictions, columns=column_names).to_csv("randomforest.csv", index_label="sample_index")
adaBoost_predicrions= adaBoost.predict(X_test)
pd.DataFrame(data=adaBoost_predicrions, columns=column_names).to_csv("adaBoost_95.csv", index_label="sample_index")